# Dynamic Levels — support / resistance / pullback

A level explorer: detects support/resistance/pullback horizontal levels from N-bar pivots + tracks their invalidation; visualizes them.

Research / visualization notebook (no backtest, no P&L). \
Detects three families of horizontal price levels and draws their lifecycle on the chart. \
Each family is seeded at pivot candles — bars that are strict extrema of their low or high relative to pivot_window neighbors on each side.

The three families (each seeded at a strict symmetric pivot, then tracked until invalidated):
- Resistance (red) — seeded at minimum lows; dies when a later candle's high comes within tolerance, or it's bracketed 'inval' times.
    - A candle whose low is strictly below the lows of pivot_window neighbors on each side. 
    - The pivot's low becomes the level price. 
    - Invalidated when a later candle's high comes within delta of the level, or the level is crossed by N candles.
- Support (green) — seeded at maximum highs; dies on a later low within tolerance, or bracketing.
- Pullback (orange) — seeded at minimum highs OR maximum lows. An "inside bar" satisfies both conditions and seeds two levels. Same invalidation rules as support, with its own delta and (typically larger) candle budget.

Look-ahead free: a pivot at bar i is only confirmed at i + pivot_window, so a level never depends on future bars.

## Configuration: automatic

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import time
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

# Candles come from the shared cache via the global ACTIVE spec
# (engine/data_configurator.py) — edit ACTIVE there to change symbol/interval/window.
from engine.data_configurator import ACTIVE, load_data, LIVE_DIR
from engine.level_detector import detect_all_levels
from engine.visualization import plot_levels
from engine.indicators import ema

import dataclasses
DATA_CONFIG = ACTIVE   # data handle; override in the manual cell below

## Configuration: manual

Per-notebook override of the data spec on top of the automatic config above. This
is a detector explorer (no strategy / exits / trade params) — the detection knobs
live in the Level Parameters chapter below, which is already the manual tuning
surface. Leave DATA_OVERRIDES empty to use the automatic ACTIVE spec.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}      # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
df = load_data(DATA_CONFIG)
print(f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m — {len(df)} bars from {df.index[0]} to {df.index[-1]}")
df.head()

## Level Parameters

Per-level delta (touch tolerance, in quote-currency points), invalidation candle counts, and pivot window (number of candles on each side of a pivot).

| Level type  | delta (pts) | invalidation candles | pivot_window |
|-------------|-------------|----------------------|--------------|
| Resistance  | 5           | 3                    | 1            |
| Support     | 15          | 3                    | 1            |
| Pullback    | 25          | 10                   | 1            |

Tune these for the symbol — a 5-point delta makes sense on BTC at ~70k but is meaningless on ETH at ~3k. A larger pivot_window (2, 3, …) yields fewer but more "structural" pivots; pivot_window=1 uses the immediate neighbor on each side.

In [ ]:
# Tolerance mode — the one knob to pick per symbol:
#   'atr'      : delta is an ATR multiple (cross-symbol; recommended default)
#   'percent'  : delta is % of the level price (cross-symbol)
#   'absolute' : delta is in quote points (symbol-specific; the source's mode)
DELTA_MODE = "atr"
ATR_PERIOD = 14            # used only when DELTA_MODE == 'atr'

# Per-family tolerances (interpreted per DELTA_MODE), invalidation budgets, and pivot windows.
# ATR-mode delta defaults below.
# For DELTA_MODE='absolute' on BTCUSDT use ~5 / 15 / 25.
# A larger pivot window = fewer, more structural pivots (a pivot at i confirms at i + window).
DELTA_RESISTANCE = 0.25
DELTA_SUPPORT    = 0.50
DELTA_PULLBACK   = 0.80

INVAL_RESISTANCE = 3
INVAL_SUPPORT    = 3
INVAL_PULLBACK   = 10

PIVOT_WINDOW_RESISTANCE = 1
PIVOT_WINDOW_SUPPORT    = 1
PIVOT_WINDOW_PULLBACK   = 1

## Detect Levels

In [ ]:
levels = detect_all_levels(
    df,
    delta_resistance=DELTA_RESISTANCE, delta_support=DELTA_SUPPORT, delta_pullback=DELTA_PULLBACK,
    inval_resistance=INVAL_RESISTANCE, inval_support=INVAL_SUPPORT, inval_pullback=INVAL_PULLBACK,
    pivot_window_resistance=PIVOT_WINDOW_RESISTANCE, pivot_window_support=PIVOT_WINDOW_SUPPORT, pivot_window_pullback=PIVOT_WINDOW_PULLBACK,
    delta_mode=DELTA_MODE, atr_period=ATR_PERIOD,
)

for kind, lst in levels.items():
    active = sum(1 for l in lst if l.invalidated_at is None)
    print(f"{kind:11}: {len(lst):4d} total | {active:4d} active | {len(lst)-active:4d} invalidated")

## Inspect

Per-level table. lifespan_candles = (invalidation candle or last candle) − seed candle. Useful for spotting levels that survived the full range vs. levels that died on the next bar.

In [ ]:
# Every level as a sortable row: when it was seeded, when (if) it invalidated,
# how long it survived, and how many candles bracketed it before dying.
last_idx = len(df) - 1
rows = []
for kind, lst in levels.items():
    for lv in lst:
        end_idx = lv.invalidated_at if lv.invalidated_at is not None else last_idx
        rows.append({
            "kind": kind,
            "price": round(lv.price, 2),
            "seed_ts": df.index[lv.start_idx],
            "invalidated_ts": df.index[lv.invalidated_at] if lv.invalidated_at is not None else None,
            "lifespan_candles": end_idx - lv.start_idx,
            "crosses": lv.cross_count,
            "active": lv.invalidated_at is None,
        })

levels_table = pd.DataFrame(rows).sort_values(["kind", "seed_ts"]).reset_index(drop=True)
levels_table.head(20)

## Visualize

Each level is drawn from its seed candle to either its invalidation candle or the right edge of the chart (for still-active levels). Toggle show_invalidated=False to declutter to active levels only.

In [ ]:
# Each level is a horizontal segment from its seed bar to its invalidation
# (or the last bar if still active). show_invalidated=False keeps the chart clean.
plot_levels(df, levels, show_invalidated=False,
            title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | dynamic levels ({DELTA_MODE})").show()

## Levels with EMAs

The same level overlay with one or more EMAs drawn on top — useful for seeing how price interacts with both horizontal S/R and a moving-average trend filter. Edit EMA_SPANS to the periods you want. EMAs are causal (bar i uses only bars 0..i), so the overlay stays look-ahead free like the levels.

In [ ]:
# Levels + user-selected EMAs on one chart. Edit EMA_SPANS to the periods you want.

EMA_SPANS = [20, 50, 200]   # EMA periods to overlay (causal: bar i uses only bars 0..i)
_EMA_COLORS = ["#3b82f6", "#f97316", "#a855f7", "#14b8a6", "#eab308"]

fig = plot_levels(df, levels, show_invalidated=False,
                  title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | levels + EMAs ({DELTA_MODE})")
for _i, _span in enumerate(EMA_SPANS):
    fig.add_trace(go.Scatter(
        x=df.index, y=ema(df["close"], _span), mode="lines",
        line=dict(color=_EMA_COLORS[_i % len(_EMA_COLORS)], width=1.3),
        name=f"EMA {_span}",
    ))
fig.show()

## Live signals

Live implementation in paper trading regime:
1. a chart opens as html & updates automatically
2. it plots a chosen ema(s) & has an option to disable its view
3. it plots only actual levels and updates them automatically (so invalidated levels disappear)

A live preview of the level map that renders in the browser and refreshes itself — the same UX as the strategy notebooks' Live signals (which run the engine), but drawing the level lifecycle instead of trades. Each poll it refetches the latest candles through the shared load_data cache (refresh=True), drops the still-forming bar, re-detects levels with the parameters above, and rewrites an auto-refreshing HTML chart under data/live/. It prints a clickable link — open it once and the tab reloads every poll. Plots only — no orders. Interrupt the cell (the stop button / Ctrl-C) to end the loop.

It reuses the detection knobs from the Level Parameters chapter and the spans from Levels with EMAs. Works best with a rolling window (a num_candles spec, or start with end left open) so the window advances; a pinned start/end range just refetches the same fixed window.

In [ ]:
# Live level map in the browser — same UX as the strategy notebooks' Live signals:
# writes an auto-refreshing HTML under data/live/ and prints a clickable link.
# Plots only — no orders. Interrupt the cell to stop.

LIVE_POLL_SECONDS = 30   # browser refresh + refetch cadence
EMA_SPANS = globals().get("EMA_SPANS", [20, 50, 200])
_EMA_COLORS = globals().get("_EMA_COLORS", ["#3b82f6", "#f97316", "#a855f7", "#14b8a6", "#eab308"])

chart_path = LIVE_DIR / f"{DATA_CONFIG.symbol}_{DATA_CONFIG.interval}_levels.html"
chart_path.parent.mkdir(parents=True, exist_ok=True)

def _write_live_levels(_df, _levels):
    fig = plot_levels(_df, _levels, show_invalidated=False,
                      title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | LIVE levels + EMAs — {_df.index[-1]:%Y-%m-%d %H:%M} UTC")
    for _i, _span in enumerate(EMA_SPANS):
        fig.add_trace(go.Scatter(x=_df.index, y=ema(_df["close"], _span), mode="lines",
                                 line=dict(color=_EMA_COLORS[_i % len(_EMA_COLORS)], width=1.3), name=f"EMA {_span}"))
    fig.write_html(str(chart_path))
    # inject a meta-refresh so the open browser tab reloads each poll (same trick build_chart uses)
    html = chart_path.read_text()
    chart_path.write_text(html.replace("<head>", f'<head><meta http-equiv="refresh" content="{LIVE_POLL_SECONDS}">', 1))

# Announce the chart once — clickable in Jupyter, file:// link otherwise (mirrors LiveEngine._announce_chart).
_uri = chart_path.resolve().as_uri()
display(HTML(f'<a href="{_uri}" target="_blank" rel="noopener">Open live level chart — {DATA_CONFIG.symbol} {DATA_CONFIG.interval}m ↗</a>'))
print(f"Live level preview → {_uri}\nThe tab auto-refreshes every {LIVE_POLL_SECONDS}s. Interrupt the cell to stop.")

try:
    while True:
        live_df = load_data(DATA_CONFIG, refresh=True).iloc[:-1]   # drop the still-forming bar (no repaint)
        live_levels = detect_all_levels(
            live_df,
            delta_resistance=DELTA_RESISTANCE, delta_support=DELTA_SUPPORT, delta_pullback=DELTA_PULLBACK,
            inval_resistance=INVAL_RESISTANCE, inval_support=INVAL_SUPPORT, inval_pullback=INVAL_PULLBACK,
            pivot_window_resistance=PIVOT_WINDOW_RESISTANCE, pivot_window_support=PIVOT_WINDOW_SUPPORT,
            pivot_window_pullback=PIVOT_WINDOW_PULLBACK,
            delta_mode=DELTA_MODE, atr_period=ATR_PERIOD,
        )
        _write_live_levels(live_df, live_levels)
        print(f"updated {live_df.index[-1]} UTC | {len(live_df)} bars | next refresh in {LIVE_POLL_SECONDS}s — interrupt to stop.")
        time.sleep(LIVE_POLL_SECONDS)
except KeyboardInterrupt:
    print("live preview stopped.")